# Pydantic model to structure output

In [7]:
# TODO:skapa en agent som ska simulera en IT-anställd
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- full name
- age
- gender
- job_title
- salary in SEK per month
    """
)



result = await employee_simulator_agent.run("simulate two employees")

In [8]:
print(result.output)

Here are two simulated employees based on the criteria:

---

**Employee 1**  
- **Full Name:** John Doe  
- **Age:** 35  
- **Gender:** Male  
- **Job Title:** Software Engineer  
- **Salary:** 6.8 million SEK (approx. 6,800 SEK/month)  

**Employee 2**  
- **Full Name:** Anna Johnson  
- **Age:** 32  
- **Gender:** Female  
- **Job Title:** Data Scientist  
- **Salary:** 6.67 million SEK (approx. 6,667 SEK/month)  

---

Both roles align with Sweden’s IT/data science trends, with salaries reflecting regional market standards. Let me know if further details are needed!


In [10]:
with open('simulated_employees.md', 'w', encoding='utf-8') as file:
    file.write(result.output)

## Get more structured output

issue with above:

 - output structure vary    
 - hard to work with the data e.g. compute mean of salaries

 want:
    - repeatable structure

In [11]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class EmployeeModel(BaseModel):
    name: str = Field(description='Mostly swedish names, but could be foreign names as well')
    age: int = Field(description= 'age should be between 18 and 67')
    gender: Literal['Male','Female']
    experience_level: Literal['Entry', 'Mid level', 'Senior', 'Expert']
    job_title: str
    salary: int = Field(gte=30_000, lte=50_000, description='salary should be between 30k and 50k, the higher experience level, the higher salary')

employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.""", retries=1
)

result= await employee_simulator_agent.run('Give me three employees', output_type=list[EmployeeModel])
result

C:\Users\Lilit\AppData\Local\Temp\ipykernel_18680\2870771557.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(gte=30_000, lte=50_000, description='salary should be between 30k and 50k, the higher experience level, the higher salary')


AgentRunResult(output=[EmployeeModel(name='Anna Svensson', age=28, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=35000), EmployeeModel(name='Erik Lindqvist', age=35, gender='Male', experience_level='Expert', job_title='AI Data Scientist', salary=45000), EmployeeModel(name='Sofia Arnbäck', age=40, gender='Female', experience_level='Senior', job_title='Machine Learning Engineer', salary=42000)])

In [14]:
result.output[0]

EmployeeModel(name='Anna Svensson', age=28, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=35000)

In [21]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Anna Svensson',
 'age': 28,
 'gender': 'Female',
 'experience_level': 'Mid level',
 'job_title': 'Data Engineer',
 'salary': 35000}

### TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on the list 

In [17]:
employees = result.output
employees

[EmployeeModel(name='Anna Svensson', age=28, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=35000),
 EmployeeModel(name='Erik Lindqvist', age=35, gender='Male', experience_level='Expert', job_title='AI Data Scientist', salary=45000),
 EmployeeModel(name='Sofia Arnbäck', age=40, gender='Female', experience_level='Senior', job_title='Machine Learning Engineer', salary=42000)]

In [19]:
employees_list = [employ.model_dump() for employ in employees]
employees_list

[{'name': 'Anna Svensson',
  'age': 28,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Data Engineer',
  'salary': 35000},
 {'name': 'Erik Lindqvist',
  'age': 35,
  'gender': 'Male',
  'experience_level': 'Expert',
  'job_title': 'AI Data Scientist',
  'salary': 45000},
 {'name': 'Sofia Arnbäck',
  'age': 40,
  'gender': 'Female',
  'experience_level': 'Senior',
  'job_title': 'Machine Learning Engineer',
  'salary': 42000}]

In [22]:
import pandas as pd

df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Anna Svensson,28,Female,Mid level,Data Engineer,35000
1,Erik Lindqvist,35,Male,Expert,AI Data Scientist,45000
2,Sofia Arnbäck,40,Female,Senior,Machine Learning Engineer,42000


In [23]:
df['salary'].mean()

np.float64(40666.666666666664)

In [33]:
df.to_csv('simulated_employees.csv', index=False)